# HydroSeason Rainfall IO Examples

This notebook shows how to read common rainfall files and run HydroSeason. The readers return the same tidy monthly schema used by the pipeline: `Date`, `Year`, `Month`, and `Rainfall_mm`.

In [ ]:
from pathlib import Path

import pandas as pd

from hydroseason import (
    delineate_monthly_dataframe,
    generate_html_report,
    read_bom_monthly,
    read_rainfall,
    read_silo,
    run_rainfall,
)

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

OUTPUT = ROOT / "output"
OUTPUT.mkdir(exist_ok=True)
ROOT

## Tidy Monthly CSV

Use `delineate_monthly_dataframe()` when your data already has monthly columns. The bundled example uses `Rainfall_mm` as the value column.

In [ ]:
csv_path = ROOT / "data" / "DATASET.csv"
df = pd.read_csv(csv_path)

artifacts = delineate_monthly_dataframe(df)
result = artifacts.result

print("Regime:", artifacts.diagnostics.regime)
print("Hydro-year start month:", artifacts.diagnostics.hydro_year_start_month)
result[["Date", "Rainfall_mm", "SeasonType", "Hydro_Year"]].head()

## Bureau of Meteorology Monthly Rainfall CSV

BoM product `IDCJAC0001` files are monthly rainfall CSVs. HydroSeason detects the long rainfall column name and returns `Rainfall_mm`. Rows where `Quality != 'Y'` are dropped by default.

In [ ]:
bom_path = ROOT / "tests" / "fixtures" / "bom_idcjac0001.csv"
bom_monthly = read_bom_monthly(bom_path)
bom_monthly.head()

## SILO Point Files

SILO fixed-format point files include metadata headers and daily or monthly data blocks. Daily rainfall is summed to monthly totals automatically. SILO custom CSV exports are also supported.

In [ ]:
silo_daily_path = ROOT / "tests" / "fixtures" / "silo_rainonly_daily.txt"
silo_daily_monthly = read_silo(silo_daily_path)
silo_daily_monthly

## One-Step Rainfall Workflow

Use `read_rainfall(source='auto')` to auto-detect BoM and SILO fixed files. Use `run_rainfall()` to read and delineate in one call.

In [ ]:
auto_df = read_rainfall(csv_path, source="auto")
auto_df.head()

In [ ]:
rainfall_artifacts = run_rainfall(
    csv_path,
    source="auto",
    output_csv=OUTPUT / "rainfall_io_results.csv",
)

print("Regime:", rainfall_artifacts.diagnostics.regime)
print("Rows:", len(rainfall_artifacts.result))

## CLI Equivalents

```bash
hydroseason rainfall --input data/DATASET.csv --source csv --output output/rainfall_results.csv
hydroseason rainfall --input IDCJAC0001_003018_Data1.csv --source auto --output output/bom_results.csv
hydroseason rainfall --input silo_rainonly.txt --source silo --output output/silo_point_results.csv
```

## Report Export

The same report tools work regardless of whether rainfall came from a local CSV, BoM, SILO point file, ERA5, or SILO polygon fetch.

In [ ]:
report_path = generate_html_report(rainfall_artifacts, OUTPUT / "rainfall_io_report.html")
report_path